# Session 6 · Understanding Sqoop and Joins in MapReduce

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/02-big-data-systems-with-hadoop/code/session-06-sqoop-and-mapreduce-joins.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

**Setup:** the Spark examples need PySpark (and Java, which Google Colab already has).

In [1]:
%pip install -q pyspark

### Example 1 · How Sqoop splits an import (simulated with SQLite)

In [2]:
import sqlite3, random, os, tempfile
random.seed(2)
con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE orders(id INTEGER PRIMARY KEY, customer_id INT, amount REAL, city TEXT)")
cities = ["Karachi", "Lahore", "Islamabad", "Peshawar", "Quetta"]
con.executemany("INSERT INTO orders VALUES (?,?,?,?)",
                [(i, random.randint(1, 50), round(random.uniform(500, 20000), 2), random.choice(cities))
                 for i in range(1, 1001)])

def sqoop_import(table, split_by, m, target):
    lo, hi = con.execute(f"SELECT MIN({split_by}), MAX({split_by}) FROM {table}").fetchone()
    print(f"BoundingValsQuery: MIN={lo}, MAX={hi}, mappers={m}")
    os.makedirs(target, exist_ok=True)
    step = (hi - lo + 1) / m
    for i in range(m):
        a, b = int(lo + i * step), int(lo + (i + 1) * step)
        where = f"{split_by} >= {a} AND {split_by} < {b}"
        rows = con.execute(f"SELECT * FROM {table} WHERE {where}").fetchall()
        with open(f"{target}/part-m-{i:05d}", "w") as f:
            f.writelines(",".join(map(str, r)) + "\n" for r in rows)
        print(f"  mapper {i}: WHERE {where:<22} -> {len(rows)} rows -> part-m-{i:05d}")

target = os.path.join(tempfile.mkdtemp(), "orders")      # stands in for --target-dir /data/orders
sqoop_import("orders", "id", 4, target)
print(sorted(os.listdir(target)))
print("first line of part-m-00000:", open(f"{target}/part-m-00000").readline().strip())

BoundingValsQuery: MIN=1, MAX=1000, mappers=4
  mapper 0: WHERE id >= 1 AND id < 251   -> 250 rows -> part-m-00000
  mapper 1: WHERE id >= 251 AND id < 501 -> 250 rows -> part-m-00001
  mapper 2: WHERE id >= 501 AND id < 751 -> 250 rows -> part-m-00002
  mapper 3: WHERE id >= 751 AND id < 1001 -> 250 rows -> part-m-00003
['part-m-00000', 'part-m-00001', 'part-m-00002', 'part-m-00003']
first line of part-m-00000: 1,4,2285.9,Islamabad


### Example 2 · Incremental append import

In [3]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE orders(id INTEGER PRIMARY KEY, amount REAL)")
con.executemany("INSERT INTO orders VALUES (?,?)", [(i, i * 10.0) for i in range(1, 101)])
last_value = 0

def incremental_import():
    global last_value
    rows = con.execute("SELECT * FROM orders WHERE id > ? ORDER BY id", (last_value,)).fetchall()
    if rows:
        last_value = rows[-1][0]
    print(f"imported {len(rows):>3} new rows; saved --last-value = {last_value}")

incremental_import()                                   # night 1: full load
con.executemany("INSERT INTO orders VALUES (?,?)", [(i, i * 10.0) for i in range(101, 131)])
incremental_import()                                   # night 2: only the 30 new rows
incremental_import()                                   # night 3: nothing new

imported 100 new rows; saved --last-value = 100
imported  30 new rows; saved --last-value = 130
imported   0 new rows; saved --last-value = 130


### Example 3 · Reduce-side join

In [4]:
from collections import defaultdict
customers = [(1, "Ayesha", "Karachi"), (2, "Bilal", "Lahore"), (3, "Sana", "Multan")]
orders = [(101, 1, 2500), (102, 2, 900), (103, 1, 4100), (104, 4, 700)]

def map_customers(r):  yield r[0], ("C", r[1], r[2])
def map_orders(r):     yield r[1], ("O", r[0], r[2])

shuffle = defaultdict(list)
for r in customers:
    for k, v in map_customers(r): shuffle[k].append(v)
for r in orders:
    for k, v in map_orders(r): shuffle[k].append(v)

print("inner join (cust_id, name, city, order_id, amount):")
for key in sorted(shuffle):
    cs = [v for v in shuffle[key] if v[0] == "C"]
    os_ = [v for v in shuffle[key] if v[0] == "O"]
    for c in cs:
        for o in os_:
            print(" ", (key, c[1], c[2], o[1], o[2]))
print("records shuffled over the network:", sum(len(v) for v in shuffle.values()))

inner join (cust_id, name, city, order_id, amount):
  (1, 'Ayesha', 'Karachi', 101, 2500)
  (1, 'Ayesha', 'Karachi', 103, 4100)
  (2, 'Bilal', 'Lahore', 102, 900)
records shuffled over the network: 7


### Example 4 · Map-side (replicated) join

In [5]:
customers = {1: ("Ayesha", "Karachi"), 2: ("Bilal", "Lahore"), 3: ("Sana", "Multan")}  # small: cached on every node
order_splits = [[(101, 1, 2500), (102, 2, 900)], [(103, 1, 4100), (104, 4, 700)]]

for i, split in enumerate(order_splits):
    for oid, cid, amt in split:            # each mapper joins locally, no shuffle at all
        if cid in customers:
            name, city = customers[cid]
            print(f"mapper {i}: {(cid, name, city, oid, amt)}")
print("records shuffled over the network: 0")

mapper 0: (1, 'Ayesha', 'Karachi', 101, 2500)
mapper 0: (2, 'Bilal', 'Lahore', 102, 900)
mapper 1: (1, 'Ayesha', 'Karachi', 103, 4100)
records shuffled over the network: 0


### Example 5 · Joins in Spark: broadcast vs sort-merge (real PySpark)

In [6]:
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[2]").appName("joins")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

customers = spark.createDataFrame([(1, "Ayesha", "Karachi"), (2, "Bilal", "Lahore"), (3, "Sana", "Multan")],
                                  ["cust_id", "name", "city"])
orders = spark.createDataFrame([(101, 1, 2500), (102, 2, 900), (103, 1, 4100), (104, 4, 700)],
                               ["order_id", "cust_id", "amount"])

orders.join(customers, "cust_id", "inner").orderBy("order_id").show()
orders.join(customers, "cust_id", "left").orderBy("order_id").show()

def join_strategy(df):
    plan = df._jdf.queryExecution().executedPlan().toString()
    return "BroadcastHashJoin" if "BroadcastHashJoin" in plan else "SortMergeJoin" if "SortMergeJoin" in plan else plan[:60]

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)       # disable automatic broadcast
print("default plan     :", join_strategy(orders.join(customers, "cust_id")))
print("with broadcast() :", join_strategy(orders.join(F.broadcast(customers), "cust_id")))

+-------+--------+------+------+-------+
|cust_id|order_id|amount|  name|   city|
+-------+--------+------+------+-------+
|      1|     101|  2500|Ayesha|Karachi|
|      2|     102|   900| Bilal| Lahore|
|      1|     103|  4100|Ayesha|Karachi|
+-------+--------+------+------+-------+



+-------+--------+------+------+-------+
|cust_id|order_id|amount|  name|   city|
+-------+--------+------+------+-------+
|      1|     101|  2500|Ayesha|Karachi|
|      2|     102|   900| Bilal| Lahore|
|      1|     103|  4100|Ayesha|Karachi|
|      4|     104|   700|  NULL|   NULL|
+-------+--------+------+------+-------+

default plan     : SortMergeJoin
with broadcast() : BroadcastHashJoin
